In [ ]:
from src.helpers.model_matrix import build_model_matrix_from_wrds

all_stocks = build_model_matrix_from_wrds(
    wrds_user="wboughattas",
    start="2020-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    use_run="last"
)

In [ ]:
from src.helpers.model_matrix import build_matrix_from_all_stocks

df = build_matrix_from_all_stocks(
    all_stocks=all_stocks,
    tickers=["AAPL", "MSFT", "TSLA", "NVDA", "GOOG", "INTL"]
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

tau_label = 0.0  # Buy/Sell/Hold labeling threshold on next-day log-return Y
tau_buy = 0.05  # SCORE > tau_buy  => BUY
tau_sell = -0.05  # SCORE < tau_sell => SELL
gamma = 1.0  # weight scaling (position size = gamma * SCORE), clip later
w_max = 0.20  # per-name weight cap (abs)
fixedWindow = True  # True = sliding window, False = expanding window
initial = 192
horizon = 60  # approx 3m validation per window
step = 20  # approx 1 month step

assert df.index.names == ["permno", "date"]
dates_all = df.index.get_level_values("date").unique().sort_values()
print(len(dates_all))
assert len(dates_all) >= initial + horizon, "Not enough dates for the rolling scheme."

In [ ]:
# Build multi-class target DIR: {-1, 0, +1}
# Y already equals lagged adj-log return(t -> t+1)
DIR = pd.Series(0, index=df.index, dtype=int)
DIR.loc[df["Y"] > tau_label] = +1
DIR.loc[df["Y"] < -tau_label] = -1

# drop ticker, Y and date-like / id (already in index)
cat_cols = [c for c in ["act_measure_lag1", "pdicity_lag1"] if c in df.columns]  # strings
num_cols = [c for c in df.columns if c not in (["ticker", "Y"] + cat_cols)]

In [ ]:
# Model: multinomial logistic with standardization (numeric) + one-hot (categorical)
ct = ColumnTransformer([
    ("num", StandardScaler(with_mean=False), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols)
], remainder="drop", sparse_threshold=1.0)

clf = LogisticRegression(
    solver="saga",
    max_iter=2000,
    n_jobs=-1,
    C=1.0
)

pipe = Pipeline([("prep", ct), ("clf", clf)])

In [ ]:
# Rolling windows (time-slice)
pred_score = pd.Series(index=df.index, dtype=float)  # SCORE = p(+1) - p(-1)
pred_dir = pd.Series(index=df.index, dtype=int)  # argmax-based class (for confusion matrix)
used_mask = pd.Series(False, index=df.index)  # rows that got validated in some window

start_pos = 0
while start_pos + initial + horizon <= len(dates_all):
    train_dates = dates_all[start_pos: start_pos + initial]
    valid_dates = dates_all[start_pos + initial: start_pos + initial + horizon]

    if fixedWindow:
        start_pos += step
    else:
        # expanding: grow initial, slide validation
        start_pos += step  # still move start of validation forward by step; training grows implicitly

    # Slice rows
    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_cols + cat_cols]
    y_tr = DIR.loc[idx_tr]

    X_va = df.loc[idx_va, num_cols + cat_cols]

    # Fit & predict
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)  # columns ordered like classes_ => [-1, 0, +1]

    # Map to SCORE and predicted direction
    clses = pipe.named_steps["clf"].classes_
    col_m1 = int(np.where(clses == -1)[0][0])
    col_0 = int(np.where(clses == 0)[0][0])
    col_p1 = int(np.where(clses == +1)[0][0])

    p_up = proba[:, col_p1]
    p_dn = proba[:, col_m1]
    score = p_up - p_dn
    yhat = clses[np.argmax(proba, axis=1)]

    # Store
    pred_score.loc[idx_va] = score
    pred_dir.loc[idx_va] = yhat
    used_mask.loc[idx_va] = True

In [ ]:
# Convert scores to signals & portfolio weights
signal = pd.Series(0, index=df.index, dtype=int)
signal.loc[pred_score > tau_buy] = +1
signal.loc[pred_score < tau_sell] = -1

weights = (gamma * pred_score).clip(lower=-w_max, upper=+w_max)  # per-name weight at time t

In [ ]:
# Portfolio daily returns and equity curve (only on validated rows)
# r_t = sum_i w_{i,t} * Y_{i,t+1}; here Y already equals next-day log return

# sum across names at each date.
valid_df = df.loc[used_mask].copy()
valid_df["score"] = pred_score.loc[used_mask]
valid_df["signal"] = signal.loc[used_mask]
valid_df["weight"] = weights.loc[used_mask]

# Per-date portfolio log-return
port_ret_by_date = (valid_df["weight"] * valid_df["Y"]).groupby(level="date").sum()
equity = np.exp(port_ret_by_date.cumsum())
equity.index.name = "date"

In [ ]:
# Classification metrics on validation set
true_dir_val = DIR.loc[used_mask]
pred_dir_val = pred_dir.loc[used_mask]

cm = confusion_matrix(true_dir_val, pred_dir_val, labels=[-1, 0, +1])
report = classification_report(
    true_dir_val, pred_dir_val,
    labels=[-1, 0, +1],
    target_names=["Down", "Flat", "Up"],
    zero_division=0
)

rf_series = df.loc[used_mask, "rf_lag1"].groupby(level="date").mean()
excess_ret = port_ret_by_date - rf_series

ann_factor = 252.0
mu_ann = excess_ret.mean() * ann_factor
sd_ann = excess_ret.std(ddof=0) * np.sqrt(ann_factor)
sharpe = mu_ann / sd_ann if sd_ann > 0 else np.nan

roll_max = equity.cummax()
dd = 1.0 - equity / roll_max
max_dd = dd.max()

In [ ]:
print("=== Validation coverage ===")
print(
    f"Validated dates: {port_ret_by_date.index.min().date()} .. {port_ret_by_date.index.max().date()}  (#days={len(port_ret_by_date)})")
print(f"Validated rows:  {used_mask.sum()} of {len(df)}")

print("\n=== Confusion Matrix (rows=true, cols=pred) ===")
print(pd.DataFrame(cm, index=["Down", "Flat", "Up"], columns=["Down", "Flat", "Up"]))

print("\n=== Classification Report ===")
print(report)

print("=== Portfolio Stats (validation) ===")
print(f"Ann. excess return: {mu_ann: .4f}")
print(f"Ann. vol (excess):  {sd_ann: .4f}")
print(f"Sharpe (excess):    {sharpe: .3f}")
print(f"Max drawdown:       {max_dd: .3%}")

print("\n=== Equity curve (head) ===")
print(equity.head())
print("\n=== Equity curve (tail) ===")
print(equity.tail())

In [ ]:
import quantstats as qs

# Strategy simple returns (datetime index)
strategy_rets = equity.pct_change().dropna()
strategy_rets.name = "YaTS"

# Collapse factors to daily Series (date index)
rf_by_date = (
    df.reset_index()[["date", "rf_lag1"]]
    .dropna()
    .groupby("date", as_index=True)["rf_lag1"]
    .mean()
    .astype(float)
    .sort_index()
)

mktrf_by_date = (
    df.reset_index()[["date", "mktrf_lag1"]]
    .dropna()
    .groupby("date", as_index=True)["mktrf_lag1"]
    .mean()
    .astype(float)
    .sort_index()
)

# align all rows to strategy dates
rf_series = rf_by_date.reindex(strategy_rets.index).ffill().bfill()
bench_rets = (mktrf_by_date + rf_by_date).reindex(strategy_rets.index).ffill().bfill()
rf_series.name = "RiskFree"
bench_rets.name = "Market"

# (encountered bug: can't pass a time-series to QS, so we convert to excess returns and then pass rf=0.0 to QS)
strategy_excess = (strategy_rets - rf_series).dropna()
bench_excess = (bench_rets - rf_series).reindex(strategy_excess.index).dropna()

# ensure both series share the exact same dates
common_idx = strategy_excess.index.intersection(bench_excess.index)
strategy_excess = strategy_excess.reindex(common_idx)
bench_excess = bench_excess.reindex(common_idx)

qs.reports.html(
    strategy_excess,
    benchmark=bench_excess.to_frame("Market"),
    rf=0.0,
    periods_per_year=252,
    output="yats_tearsheet.html",
    title="Yet another Trading Strategy (YaTS)"
)